# Draw Fed-MOPSD client-student evaluation results

This standalone notebook reads the completed Math, IF, and Code `metrics.json` files. It does **not** load any model or rerun evaluation.

In [ ]:
# Papermill parameters
RESULT_DIR = "results/fed_mopsd_qwen3_4b_base/client_student_evaluation_vllm_38912"
CHART_FILENAME = "fed_mopsd_client_students_scores.png"
CHART_TITLE = "Fed-MOPSD Qwen3-4B: Client Students Evaluation"
Y_MAX = 108.0


In [ ]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

workspace = Path.cwd().resolve()
result_dir = Path(RESULT_DIR).expanduser()
if not result_dir.is_absolute():
    result_dir = (workspace / result_dir).resolve()

domain_metric_files = {
    domain: result_dir / domain / "metrics.json"
    for domain in ("math", "if", "code")
}
missing = [str(path) for path in domain_metric_files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing completed evaluation metrics:\n  - " + "\n  - ".join(missing)
    )

print(f"Workspace:  {workspace}", flush=True)
print(f"Result dir: {result_dir}", flush=True)
for domain, path in domain_metric_files.items():
    print(f"[FOUND] {domain}: {path}", flush=True)


In [ ]:
all_metrics = []
domain_payloads = {}
for domain, path in domain_metric_files.items():
    payload = json.loads(path.read_text(encoding="utf-8-sig"))
    rows = payload.get("metrics")
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"No metric rows found in {path}")
    for row in rows:
        normalized = dict(row)
        normalized.setdefault("domain", domain)
        all_metrics.append(normalized)
    domain_payloads[domain] = payload

metrics = pd.DataFrame(all_metrics)
required_columns = {"benchmark", "domain", "metric", "score_percent"}
missing_columns = sorted(required_columns - set(metrics.columns))
if missing_columns:
    raise ValueError(f"Metric files are missing columns: {missing_columns}")

combined_csv = result_dir / "client_students_vllm_metrics.csv"
combined_json = result_dir / "client_students_vllm_metrics.json"
metrics.to_csv(combined_csv, index=False, encoding="utf-8-sig")
combined_json.write_text(
    json.dumps(
        {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "domains": domain_payloads,
            "metrics": all_metrics,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"[SAVED] {combined_csv}", flush=True)
print(f"[SAVED] {combined_json}", flush=True)
display(metrics)


In [ ]:
plot_specs = [
    {"benchmark": "AIME25", "metric": "avg_at_12", "label": "AIME25\nAvg@12", "color": "#4C78A8"},
    {"benchmark": "AIME26", "metric": "avg_at_12", "label": "AIME26\nAvg@12", "color": "#4C78A8"},
    {"benchmark": "IFBENCH", "metric": "prompt_level_loose", "label": "IFBench\nPrompt loose", "color": "#F58518"},
    {"benchmark": "IFEVAL", "metric": "prompt_level_loose", "label": "IFEval\nPrompt loose", "color": "#F58518"},
    {"benchmark": "LiveCodeBench", "metric": "pass@1", "label": "LiveCodeBench\nPass@1", "color": "#54A24B"},
]

selected_rows = []
for spec in plot_specs:
    matched = metrics[
        (metrics["benchmark"] == spec["benchmark"])
        & (metrics["metric"] == spec["metric"])
    ]
    if len(matched) != 1:
        raise ValueError(
            f"Expected exactly one {spec['benchmark']}/{spec['metric']} row; found {len(matched)}"
        )
    row = matched.iloc[0].to_dict()
    score_percent = float(row["score_percent"])
    if not math.isfinite(score_percent) or not 0.0 <= score_percent <= 100.0:
        raise ValueError(
            f"Invalid score_percent for {spec['benchmark']}/{spec['metric']}: {score_percent}"
        )
    if "status" in row and str(row["status"]).lower() != "complete":
        raise ValueError(
            f"Incomplete metric for {spec['benchmark']}/{spec['metric']}: {row['status']}"
        )
    selected_rows.append({**spec, **row, "score_percent": score_percent})

primary_metrics = pd.DataFrame(selected_rows)
display(
    primary_metrics[["benchmark", "domain", "metric", "score_percent", "numerator", "denominator"]]
)


In [ ]:
labels = primary_metrics["label"].tolist()
values = primary_metrics["score_percent"].astype(float).tolist()
colors = primary_metrics["color"].tolist()

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
figure, axis = plt.subplots(figsize=(14, 7.5))
bars = axis.bar(
    range(len(labels)),
    values,
    width=0.68,
    color=colors,
    edgecolor="#333333",
    linewidth=0.8,
    zorder=3,
)

for bar, value in zip(bars, values):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.2,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="semibold",
    )

axis.set_xticks(range(len(labels)), labels)
axis.tick_params(axis="x", labelsize=12, pad=8)
axis.tick_params(axis="y", labelsize=11)
axis.set_ylabel("Score (%)", fontsize=13)
axis.set_ylim(0, float(Y_MAX))
axis.set_title(CHART_TITLE, fontsize=16, pad=16)
axis.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.35, zorder=0)
axis.set_axisbelow(True)
figure.tight_layout()

chart_path = result_dir / CHART_FILENAME
svg_path = chart_path.with_suffix(".svg")
pdf_path = chart_path.with_suffix(".pdf")
figure.savefig(chart_path, dpi=300, bbox_inches="tight", facecolor="white")
figure.savefig(svg_path, bbox_inches="tight", facecolor="white")
figure.savefig(pdf_path, bbox_inches="tight", facecolor="white")
display(figure)
plt.close(figure)

print(f"[SAVED] PNG: {chart_path}", flush=True)
print(f"[SAVED] SVG: {svg_path}", flush=True)
print(f"[SAVED] PDF: {pdf_path}", flush=True)
print("[DONE] Client-student result drawing completed; no evaluation was rerun.", flush=True)


## Papermill command

```bash
cd /workspace/code

papermill \
  eval_client_students_vllm_draw.ipynb \
  eval_client_students_vllm_draw_running.ipynb \
  --cwd /workspace/code \
  --request-save-on-cell-execute \
  --log-output \
  > eval_client_students_vllm_draw.log 2>&1
```

No GPU environment variable is required.